In [1]:
from pathlib import Path
while not (Path.cwd() / '.git').exists():
    %cd ..

/home/matthew/study/grab-voc-triage


In [12]:
import pandas as pd
import config
import numpy as np
from sklearn import metrics

In [4]:
df_mini = pd.read_csv("data/processed/manual_annotation_openai_annotated.csv")
df_mini.head(2)

,Unnamed: 0.1,Unnamed: 0,userName,score,at,content,DRIVER_OPS,APP_AND_MAPS,PRICING_AND_BILLING,FULFILLMENT_FOOD,DRIVER_OPS.1,APP_AND_MAPS.1,PRICING_AND_BILLING.1,FULFILLMENT_FOOD.1
0,0,141,Pengguna Google,1,2026-08-29 11:18:30,gak tau kenapa udah gak kayak dulu lagi pakai ...,NEG,ABSENT,ABSENT,ABSENT,ABSENT,ABSENT,ABSENT,NEG
1,1,320,Pengguna Google,1,2026-08-22 00:49:40,skrang grab jelek banget driver nya gapernah d...,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT,ABSENT,NEG


In [9]:
df_4o = pd.read_csv("data/processed/manual_annotation_gpt4o.csv")
df_4o

,Unnamed: 0,userName,score,at,content,DRIVER_OPS,APP_AND_MAPS,PRICING_AND_BILLING,FULFILLMENT_FOOD,DRIVER_OPS.1,APP_AND_MAPS.1,PRICING_AND_BILLING.1,FULFILLMENT_FOOD.1
0,141,Pengguna Google,1,2026-08-29 11:18:30,gak tau kenapa udah gak kayak dulu lagi pakai ...,NEG,ABSENT,ABSENT,ABSENT,ABSENT,ABSENT,NEG,ABSENT
1,320,Pengguna Google,1,2026-08-22 00:49:40,skrang grab jelek banget driver nya gapernah d...,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT,ABSENT,ABSENT
2,1335,Pengguna Google,1,2026-07-15 11:54:04,driver nya gak jujur jelas naro titik sudah be...,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT,ABSENT,ABSENT
3,1496,Pengguna Google,1,2026-07-08 13:57:12,knpa sya ksi bintang satu karna ada kurir kura...,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT,NEG,ABSENT
4,2216,Pengguna Google,1,2026-06-09 16:32:58,jelek banget pelayanannya kurang dan sering bu...,ABSENT,NEG,NEG,ABSENT,ABSENT,NEG,NEG,ABSENT
5,541,Pengguna Google,1,2026-08-14 16:51:14,pembaruan makin ke sini makin memburuk terutam...,ABSENT,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT,ABSENT
6,2067,Pengguna Google,1,2026-06-20 17:46:25,sekarang masukin no hp pun eroor apalah ini,ABSENT,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT,ABSENT
7,480,Pengguna Google,1,2026-08-16 19:56:44,lucu deh ni apk ga usah ada option hemat kalo ...,NEG,ABSENT,NEG,ABSENT,ABSENT,NEG,ABSENT,ABSENT
8,3248,Dendo Se,1,2026-05-02 15:56:01,wkwkwk sangat2 tidak worth it aplikasi nya ove...,ABSENT,ABSENT,NEG,ABSENT,ABSENT,NEG,NEG,ABSENT
9,205,Pengguna Google,1,2026-08-26 17:40:05,pdhl alamat udh sesuai titik tpi mlh app nya n...,ABSENT,NEG,ABSENT,ABSENT,ABSENT,NEG,ABSENT,ABSENT


In [32]:
def classification_report(y_true : np.array, y_pred : np.array) -> str:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    print('--------------------------')
    print('PRIMARY METRIC')
    print('--------------------------')
    for i, cat in enumerate(config.CATEGORIES):
        macro_f1 = metrics.f1_score(y_true[:, i], y_pred[:, i], average = 'macro')
        print(f"{cat} Macro F1-score: {macro_f1}")
    print('--------------------------')
    print('SECONDARY METRIC')
    print('--------------------------')
    print((y_true == y_pred).all(axis = 1).mean())
    print('--------------------------')
    print('DIAGNOSTIC METRIC')
    print('--------------------------')
    for i, cat in enumerate(config.CATEGORIES):
        recalls = metrics.recall_score(y_true[:, i], y_pred[:, i], average = None, labels = ['POS', 'ABSENT', 'NEG'])
        print(f"{cat} POS Recall: {recalls[0]}")
        print(f"{cat} ABSENT Recall: {recalls[1]}")
        print(f"{cat} NEG Recall: {recalls[2]}")
    
classification_report(df_mini[config.CATEGORIES], df_mini[np.array(config.CATEGORIES) + ".1"])
print()
classification_report(df_4o[config.CATEGORIES], df_4o[np.array(config.CATEGORIES) + ".1"])

--------------------------
PRIMARY METRIC
--------------------------
DRIVER_OPS Macro F1-score: 0.8222222222222223
APP_AND_MAPS Macro F1-score: 0.5583333333333333
PRICING_AND_BILLING Macro F1-score: 0.891471048513302
FULFILLMENT_FOOD Macro F1-score: 0.7114695340501792
--------------------------
SECONDARY METRIC
--------------------------
0.72
--------------------------
DIAGNOSTIC METRIC
--------------------------
DRIVER_OPS POS Recall: 0.6
DRIVER_OPS ABSENT Recall: 0.9705882352941176
DRIVER_OPS NEG Recall: 0.7272727272727273
APP_AND_MAPS POS Recall: 0.0
APP_AND_MAPS ABSENT Recall: 0.8235294117647058
APP_AND_MAPS NEG Recall: 0.9333333333333333
PRICING_AND_BILLING POS Recall: 0.6666666666666666
PRICING_AND_BILLING ABSENT Recall: 0.9714285714285714
PRICING_AND_BILLING NEG Recall: 0.9166666666666666
FULFILLMENT_FOOD POS Recall: 1.0
FULFILLMENT_FOOD ABSENT Recall: 0.9375
FULFILLMENT_FOOD NEG Recall: 1.0

--------------------------
PRIMARY METRIC
--------------------------
DRIVER_OPS Macro F

In [34]:
for c in config.CATEGORIES:
    print(df_4o[c].value_counts())

DRIVER_OPS
ABSENT    34
NEG       11
POS        5
Name: count, dtype: int64
APP_AND_MAPS
ABSENT    34
NEG       15
POS        1
Name: count, dtype: int64
PRICING_AND_BILLING
ABSENT    35
NEG       12
POS        3
Name: count, dtype: int64
FULFILLMENT_FOOD
ABSENT    48
NEG        1
POS        1
Name: count, dtype: int64


based on the data distribution, food category is very small hence will be dropped. Furthermore, the POS also come to a very small percentage of the data. 
Cause generally the positive comment is on overall generic stuff, they mostly clustered to ABSENT. From the business perspective, NEG is much more informative to fix problem in the company. 